# Wildfire event mapping using Sentinel-2 data

Map the extent of wildfires and classify their severity using Sentinel-2 data.

Generate a pre event mosaic and use the first available post event image. 
Calculate the NBR for both and generate a change image using the difference between pre and post event and apply thresholds to classify the severity of the areas affected by the fire. 

In [6]:
import openeo
import rasterio
from openeo.processes import ProcessBuilder
from openeo.processes import quantiles
from folium.plugins import Draw
import leafmap
from IPython.display import JSON
from shapely.geometry import shape

In [3]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/fire/Fontainebleau")
#out_dir.mkdir()
event = "Fontainebleau_2026"

BANDS = ["B02", "B03", "B04", "B08", "B12","SCL"]  

## 1) Define time frame and extent

In [4]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [7]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 2.756195, 'south': 48.268112, 'east': 2.756882, 'north': 48.268112}


In [ ]:
# dates stelvio fire
PRE_DATE  = ("2024-07-01", "2024-07-31")   
POST_DATE = ("2025-06-15", "2025-06-20")   

In [ ]:
# dates tuscany fire
PRE_DATE  = ("2025-05-01", "2025-05-30")   
POST_DATE = ("2026-05-27", "2026-05-30")   

In [12]:
# dates fontainebleau fire
PRE_DATE  = ("2025-07-01", "2025-07-30")   
POST_DATE = ("2026-07-20", "2026-07-25")   

### 2) Authentificate and load the pre and post Sentinel-2 cube

In [9]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

Authenticated using refresh token.
Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [10]:
# Load Sentinel-2 data
def load_s2(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=20,
    )
    return cube.resample_spatial(resolution=10, method="bilinear")

pre_cube=load_s2(PRE_DATE)
post_cube = load_s2(POST_DATE)

### 3) Create the mask from SCL

Create the mask using the following SCL values:

```
1  = SC_SATURATED_DEFECTIVE
3  = SC_CLOUD_SHADOW
7  = SC_CLOUD_LOW_PROBA / UNCLASSIFIED
8  = SC_CLOUD_MEDIUM_PROBA
9  = SC_CLOUD_HIGH_PROBA
10 = SC_THIN_CIRRUS
```

In [11]:
def mask_clouds(cube):
    scl = cube.band("SCL")
    return (
    (scl == 1) |
    (scl == 3) |
    (scl == 7) |
    (scl == 8) |
    (scl == 9) |
    (scl == 10)
    )

### 4) Generate the layer of valid observation

In [13]:
post_cube_mask = mask_clouds(post_cube)
post_cube_masked = post_cube.mask(post_cube_mask)

In [14]:
file_post_cube = out_dir/ f"post_{event}.tif"

print(file_post_cube)

post_cube_masked.download(file_post_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/fire/Fontainebleau/post_Fontainebleau_2026.tif


### 5) Generate monthly mosaic for pre event 

In [15]:
pre_cube_mask = mask_clouds(pre_cube)

pre_cube_masked = pre_cube.mask(pre_cube_mask).reduce_dimension(
    dimension='t',
    reducer=lambda x: quantiles(data=x, probabilities=[0.25])
)

In [28]:
file_pre_cube = out_dir/ f"pre_mosaic_{event}.tif"

print(file_pre_cube)

pre_cube_masked.download(file_pre_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/fire/Fontainebleau/pre_mosaic_Fontainebleau_2026.tif


### 6) Calculate NBR for pre and post event

In [11]:
def compute_nbr(cube):
    swir = cube.band("B12")
    nir   = cube.band("B08")
    return (nir - swir) / (nir + swir)

pre_nbr  = compute_nbr(pre_cube_masked)
post_nbr = compute_nbr(post_cube_masked)

In [12]:
file_pre_nbr = out_dir/ f"pre_mosaic_{event}_NBR.tif"
file_post_nbr = out_dir/ f"post_{event}_NBR.tif"

pre_nbr.download(file_pre_nbr)
post_nbr.download(file_post_nbr)

### 7) Calculate simple ratio between pre and post event

In [14]:
from openeo.processes import subtract

# simple ratio
ratio_nbr = pre_nbr.merge_cubes(post_nbr, overlap_resolver=subtract)

file_ratio = out_dir/ f"ratio_{event}_NBR.tif"

ratio_nbr.download(file_ratio)

### 8) Classify severity of the fire affected areas

Thresholds are defined by Key, C.H., Benson, N.C., 2006. Landscape Assessment (LA). In: Lutes, D.C., Keane, R.E., Caratti, J.F., Key, C.H., Benson, N.C., Sutherland, S., Gangi, L.J. (Eds.), FIREMON: Fire Effects Monitoring and Inventory System

In [ ]:
def classify_nbr(ratio_nbr):
    return (
        (ratio_nbr > 0.66) * 4 +
        ((ratio_nbr > 0.44) & (ratio_nbr <= 0.66)) * 3 +
        ((ratio_nbr > 0.27) & (ratio_nbr <= 0.44)) * 2 +
        ((ratio_nbr > 0.1) & (ratio_nbr <= 0.27)) * 1
        #else 0 = unburned
    )

severity_class = classify_nbr(ratio_nbr)
severity_class.download(out_dir / f"severity_class_{event}_nbr.tif")